In [ ]:
# Run this cell first to install required packages in JupyterLite
%pip install -q numpy matplotlib ipywidgets scikit-learn


# Figure 17.2: Illustration of PCA for dimensionality reduction
**Principal Component Analysis (PCA)** finds the directions (principal components) of maximum variance in data.

**Covariance matrix and eigendecomposition:**

$$C = \frac{1}{N}X^TX, \quad C v_k = \lambda_k v_k$$

- $C \in \mathbb{R}^{d \times d}$ is the (centred) sample covariance matrix.
- $\lambda_k$ is the $k$-th eigenvalue — the variance explained along the $k$-th PC.
- $v_k$ is the $k$-th eigenvector — the direction of the $k$-th PC.

**Explained variance:** the fraction of total variance captured by the top-$k$ PCs is
$$\text{Explained}(k) = \frac{\sum_{i=1}^k \lambda_i}{\sum_{i=1}^d \lambda_i}$$

**Connection to linear autoencoders:**
A linear autoencoder without bias learns the same k-dimensional subspace as the top-k PCA components.
Minimising reconstruction MSE with a bottleneck of size $k$ is equivalent to projecting onto the top-$k$ eigenvectors of $C$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

def draw_pca(n_points=200, angle_deg=30, var_ratio=10, n_components=2):
    rng = np.random.default_rng(42)
    theta = angle_deg * np.pi / 180
    R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    cov = R @ np.diag([float(var_ratio), 1.0]) @ R.T
    X = rng.multivariate_normal([0, 0], cov, n_points)
    # Center
    X -= X.mean(axis=0)
    # PCA via covariance
    C = X.T @ X / n_points
    eigenvalues, eigenvectors = np.linalg.eigh(C)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    explained = eigenvalues / eigenvalues.sum() * 100

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(X[:, 0], X[:, 1], alpha=0.4, s=15, color='#94a3b8', label='Data')
    colors = ['#2563eb', '#dc2626']
    for i in range(n_components):
        scale = np.sqrt(eigenvalues[i]) * 2
        vec = eigenvectors[:, i] * scale
        axes[0].annotate('', xy=(vec[0], vec[1]), xytext=(0, 0),
                         arrowprops=dict(arrowstyle='->', color=colors[i], lw=2.5))
        axes[0].annotate('', xy=(-vec[0], -vec[1]), xytext=(0, 0),
                         arrowprops=dict(arrowstyle='->', color=colors[i], lw=2.5))
        axes[0].plot([], [], color=colors[i], lw=2, label=f'PC{i+1} ({explained[i]:.1f}%)')
    axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)
    axes[0].set_title('Data + Principal Components', fontsize=11)
    axes[0].legend(fontsize=9)

    axes[1].barh(['PC2', 'PC1'], [explained[1], explained[0]], color=['#dc2626', '#2563eb'])
    axes[1].set_xlabel('Explained variance (%)'); axes[1].set_xlim(0, 100)
    axes[1].set_title('Explained Variance', fontsize=11)
    for i, v in enumerate([explained[1], explained[0]]):
        axes[1].text(v + 1, i, f'{v:.1f}%', va='center', fontsize=10)

    plt.suptitle(f'PCA: θ={angle_deg}°, σ₁/σ₂={var_ratio}, N={n_points}', fontsize=10, y=0.0)
    plt.tight_layout(); plt.show()
    print(f'Eigenvalues: λ₁={eigenvalues[0]:.3f}, λ₂={eigenvalues[1]:.3f}')
    print(f'Explained: PC1={explained[0]:.1f}%, PC2={explained[1]:.1f}%')
    print(f'Linear AE with 1 neuron ≈ projecting onto PC1 only')

widgets.interact(draw_pca,
    n_points=widgets.IntSlider(min=50, max=500, step=50, value=200, description='N points', continuous_update=False),
    angle_deg=widgets.IntSlider(min=0, max=90, step=5, value=30, description='Angle θ°', continuous_update=False),
    var_ratio=widgets.IntSlider(min=1, max=20, step=1, value=10, description='Variance ratio', continuous_update=False),
    n_components=widgets.Dropdown(options=[('1 PC', 1), ('2 PCs', 2)], value=2, description='Show PCs'),
)